# Build VITALS Flat File from MIMIC-IV

This notebook demonstrates how to extract and construct the **VITALS.csv** flat file required by the **Sepy2.0 pipeline** using multiple MIMIC-IV tables.

Input sources:
- `csv_concepts_exports/vitalsign.csv`
- `csv_concepts_exports/oxygen_delivery.csv`
- `icu_chartevents.csv` + `icu_d_items.csv`
- `hosp_omr.csv`
- `icu_icustays.csv` (to add hadm_id)

Output:
- `mimic_flat_files/VITALS.csv`


In [3]:
# Step 0. Setup environment
import pandas as pd
import numpy as np
import os
import re

base_dir_raw = "/hpc/group/kamaleswaranlab/mimic_iv/builtdata/csv_exports"
base_dir_concepts = "/hpc/group/kamaleswaranlab/mimic_iv/builtdata/csv_concepts_exports"
output_dir = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"

os.makedirs(output_dir, exist_ok=True)

print("✅ Environment ready")

✅ Environment ready


# Step 1. Load core tables

In [4]:
# Concepts vitalsign + oxygen_delivery
vitals_concepts = pd.read_csv(os.path.join(base_dir_concepts, "vitalsign.csv"))
oxygen_concepts = pd.read_csv(os.path.join(base_dir_concepts, "oxygen_delivery.csv"))

# ICU icustays (to add hadm_id)
icustays = pd.read_csv(os.path.join(base_dir_raw, "icu_icustays.csv"))

# ICU chartevents (read in chunks later) + d_items
d_items = pd.read_csv(os.path.join(base_dir_raw, "icu_d_items.csv"))

# OMR table
omr = pd.read_csv(os.path.join(base_dir_raw, "hosp_omr.csv"))

print("✅ Data loaded")
print("Vitalsign columns:", vitals_concepts.columns.tolist()[:10])
print("Oxygen delivery columns:", oxygen_concepts.columns.tolist()[:10])
print("ICU stays columns:", icustays.columns.tolist())
print("OMR columns:", omr.columns.tolist())

/tmp/ipykernel_101105/3936740344.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  oxygen_concepts = pd.read_csv(os.path.join(base_dir_concepts, "oxygen_delivery.csv"))


✅ Data loaded
Vitalsign columns: ['subject_id', 'stay_id', 'charttime', 'heart_rate', 'sbp', 'dbp', 'mbp', 'sbp_ni', 'dbp_ni', 'mbp_ni']
Oxygen delivery columns: ['subject_id', 'stay_id', 'charttime', 'o2_flow', 'o2_flow_additional', 'o2_delivery_device_1', 'o2_delivery_device_2', 'o2_delivery_device_3', 'o2_delivery_device_4']
ICU stays columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']
OMR columns: ['subject_id', 'chartdate', 'seq_num', 'result_name', 'result_value']


## Step 2. Basic vitals (vitalsign.csv)

Field mapping:
- `heart_rate` → `pulse`
- `resp_rate` → `unassisted_resp_rate`
- `spo2` → `spo2`
- `temperature` → `temperature`
- `temperature_site` → `temproute`
- `sbp, dbp, mbp` → `sbp_line, dbp_line, map_line`
- `sbp_ni, dbp_ni, mbp_ni` → `sbp_cuff, dbp_cuff, map_cuff`


In [5]:
vitals_base = vitals_concepts.rename(columns={
    "heart_rate": "pulse",
    "resp_rate": "unassisted_resp_rate",
    "temperature": "temperature",
    "temperature_site": "temproute",
    "sbp": "sbp_line",
    "dbp": "dbp_line",
    "mbp": "map_line",
    "sbp_ni": "sbp_cuff",
    "dbp_ni": "dbp_cuff",
    "mbp_ni": "map_cuff"
})

vitals_base = vitals_base[[
    "subject_id", "stay_id", "charttime",
    "pulse", "unassisted_resp_rate", "spo2",
    "temperature", "temproute",
    "sbp_line", "dbp_line", "map_line",
    "sbp_cuff", "dbp_cuff", "map_cuff", "glucose"
]]

print("✅ Basic vitals processed")
vitals_base.head()

✅ Basic vitals processed


,subject_id,stay_id,charttime,pulse,unassisted_resp_rate,spo2,temperature,temproute,sbp_line,dbp_line,map_line,sbp_cuff,dbp_cuff,map_cuff,glucose
0,10000032,39553978,2180-07-23 14:00:00,NaN,NaN,NaN,37.06,Oral,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10000032,39553978,2180-07-23 14:11:00,NaN,NaN,NaN,NaN,NaN,84.0,48.0,56.0,84.0,48.0,56.0,NaN
2,10000032,39553978,2180-07-23 14:12:00,91.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10000032,39553978,2180-07-23 14:13:00,NaN,NaN,98.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10000032,39553978,2180-07-23 14:30:00,93.0,21.0,97.0,NaN,NaN,95.0,59.0,67.0,95.0,59.0,67.0,NaN


## Step 3. Oxygen related (oxygen_delivery.csv)

Field mapping:
- `o2_delivery_device_1` → `o2_device`
- `o2_flow` → `o2_flow_rate`


In [6]:
oxygen = oxygen_concepts.rename(columns={
    "o2_flow": "o2_flow_rate"
})[["subject_id", "stay_id", "charttime", "o2_delivery_device_1", "o2_delivery_device_2", "o2_delivery_device_3", "o2_delivery_device_4", "o2_flow_rate"]]

print("✅ Oxygen data processed")
oxygen.head()

✅ Oxygen data processed


,subject_id,stay_id,charttime,o2_delivery_device_1,o2_delivery_device_2,o2_delivery_device_3,o2_delivery_device_4,o2_flow_rate
0,10000032,39553978,2180-07-23 14:20:00,Nasal cannula,NaN,NaN,NaN,2.0
1,10000032,39553978,2180-07-23 18:00:00,Nasal cannula,NaN,NaN,NaN,2.0
2,10000032,39553978,2180-07-23 20:00:00,Nasal cannula,NaN,NaN,NaN,2.0
3,10000690,37081114,2150-11-02 19:40:00,Non-rebreather,NaN,NaN,NaN,NaN
4,10000690,37081114,2150-11-03 08:00:00,High flow neb,NaN,NaN,NaN,10.0


## Step 4. ICU specific items (chartevents + d_items)

Itemids of interest:
- 224639, 226512, 226531 → Weight
- 226730 → Height
- 220074 → CVP
- 228640 → EtCO2


In [7]:
wanted_items = [224639, 226512, 226531, 226730, 220074, 228640]
item_labels = d_items[d_items["itemid"].isin(wanted_items)][["itemid", "label"]]
item_labels

,itemid,label
19,220074,Central Venous Pressure
766,224639,Daily Weight
1856,226512,Admission Weight (Kg)
1866,226531,Admission Weight (lbs.)
1945,226730,Height (cm)
3087,228640,EtCO2


In [8]:
chartevents_path = os.path.join(base_dir_raw, "icu_chartevents.csv")
chunksize = 1_000_000
frames = []

for chunk in pd.read_csv(chartevents_path, chunksize=chunksize):
    chunk = chunk[chunk["itemid"].isin(wanted_items)]
    frames.append(chunk)

chartevents_filtered = pd.concat(frames, ignore_index=True)

chartevents_filtered = chartevents_filtered.merge(item_labels, on="itemid", how="left")

print("✅ ICU specific items extracted:", chartevents_filtered.shape)
chartevents_filtered.head()


✅ ICU specific items extracted: (1739034, 12)


,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning,label
0,10000032,29079034,39553978,18704.0,2180-07-23 12:36:00,2180-07-23 14:45:00,226512,39.4,39.4,kg,0.0,Admission Weight (Kg)
1,10000032,29079034,39553978,18704.0,2180-07-23 12:36:00,2180-07-23 14:45:00,226730,152,152.0,cm,0.0,Height (cm)
2,10000032,29079034,39553978,18704.0,2180-07-23 14:22:00,2180-07-23 14:23:00,226531,86.7,86.7,NaN,0.0,Admission Weight (lbs.)
3,10000032,29079034,39553978,18704.0,2180-07-23 14:44:00,2180-07-23 14:45:00,226531,86.7,86.7,NaN,0.0,Admission Weight (lbs.)
4,10000690,25860671,37081114,8787.0,2150-11-06 08:00:00,2150-11-06 09:08:00,226531,121.6,121.6,NaN,0.0,Admission Weight (lbs.)


## Step 5. OMR table (hosp_omr.csv)

- Weight (lbs) → convert to kg
- Height (inch) → convert to cm
- Blood pressure string “110/70” → split into sbp_cuff, dbp_cuff, compute map_cuff


In [9]:
omr_bp = omr[omr["result_name"] == "Blood Pressure"].copy()
omr_bp["sbp_cuff"] = omr_bp["result_value"].str.split("/").str[0].astype(float)
omr_bp["dbp_cuff"] = omr_bp["result_value"].str.split("/").str[1].astype(float)
omr_bp["map_cuff"] = (omr_bp["sbp_cuff"] + 2*omr_bp["dbp_cuff"]) / 3

omr_weight = omr[omr["result_name"].str.contains("Weight")].copy()
omr_weight["daily_weight_kg"] = omr_weight["result_value"].astype(float) * 0.453592

omr_height = omr[omr["result_name"].str.contains("Height")].copy()
omr_height["result_value"] = pd.to_numeric(omr_height["result_value"], errors="coerce")
omr_height["height_cm"] = omr_height["result_value"] * 2.54

print("✅ OMR processed")
omr_bp.head()

✅ OMR processed


,subject_id,chartdate,seq_num,result_name,result_value,sbp_cuff,dbp_cuff,map_cuff
0,10000032,2180-04-27,1,Blood Pressure,110/65,110.0,65.0,80.000000
21,10000032,2180-05-25,1,Blood Pressure,106/60,106.0,60.0,75.333333
24,10000032,2180-06-01,1,Blood Pressure,121/77,121.0,77.0,91.666667
27,10000032,2180-06-22,1,Blood Pressure,100/60,100.0,60.0,73.333333
33,10000032,2180-06-30,1,Blood Pressure,102/60,102.0,60.0,74.000000


In [10]:
print(omr_bp.head())
print(omr_weight.head())
print(omr_height.head())

    subject_id   chartdate  seq_num     result_name result_value  sbp_cuff  \
0     10000032  2180-04-27        1  Blood Pressure       110/65     110.0   
21    10000032  2180-05-25        1  Blood Pressure       106/60     106.0   
24    10000032  2180-06-01        1  Blood Pressure       121/77     121.0   
27    10000032  2180-06-22        1  Blood Pressure       100/60     100.0   
33    10000032  2180-06-30        1  Blood Pressure       102/60     102.0   

    dbp_cuff   map_cuff  
0       65.0  80.000000  
21      60.0  75.333333  
24      77.0  91.666667  
27      60.0  73.333333  
33      60.0  74.000000  
   subject_id   chartdate  seq_num   result_name result_value  daily_weight_kg
1    10000032  2180-04-27        1  Weight (Lbs)           94        42.637648
4    10000032  2180-05-07        1  Weight (Lbs)        92.15        41.798503
5    10000032  2180-05-07        2  Weight (Lbs)        92.15        41.798503
6    10000032  2180-05-07        3  Weight (Lbs)        92.

## Step 6. Merge all sources into one VITALS table

We need to combine:
1. **Vitals base (vitalsign.csv)**: core vital signs
2. **Oxygen (oxygen_delivery.csv)**: oxygen device & flow
3. **Chartevents filtered (icu_chartevents.csv)**: CVP, EtCO₂, weight, height
4. **OMR (hosp_omr.csv)**: outpatient/ward blood pressure, weight, height
5. **ICU stays (icu_icustays.csv)**: to add `hadm_id` → `csn`

Final unified table will include:  
`pat_id`, `csn`, `stay_id`, `recorded_time`, and all vital signs.

### Step 6.1 Merge vitals_base and oxygen

In [11]:
# Merge core vitals and oxygen by subject_id + stay_id + charttime
vitals_merged = vitals_base.merge(
    oxygen,
    on=["subject_id", "stay_id", "charttime"],
    how="outer"
)

print("After merging vitals_base + oxygen:", vitals_merged.shape)

After merging vitals_base + oxygen: (13647548, 20)


### Step 6.2 Pivot chartevents_filtered

In [12]:
# Pivot chartevents
chartevents_pivot = chartevents_filtered.pivot_table(
    index=["subject_id", "stay_id", "charttime"],
    columns="label",
    values="valuenum",
    aggfunc="mean"
).reset_index()

# Convert Admission Weight (lbs.) to kg
if "Admission Weight (lbs.)" in chartevents_pivot.columns:
    chartevents_pivot["Admission Weight (lbs.)"] = (
        chartevents_pivot["Admission Weight (lbs.)"] * 0.4536
    )

# Merge three weight sources into one column
chartevents_pivot["daily_weight_kg"] = (
    chartevents_pivot.get("Daily Weight")
    .combine_first(chartevents_pivot.get("Admission Weight (Kg)"))
    .combine_first(chartevents_pivot.get("Admission Weight (lbs.)"))
)

# Rename other items
chartevents_pivot = chartevents_pivot.rename(columns={
    "Central Venous Pressure": "cvp",
    "EtCO2": "end_tidal_co2",
    "Height (cm)": "height_cm"
})

# Drop redundant weight columns
chartevents_pivot = chartevents_pivot.drop(
    columns=[c for c in ["Daily Weight", "Admission Weight (Kg)", "Admission Weight (lbs.)"] if c in chartevents_pivot.columns]
)

print("✅ Chartevents pivot with unified daily_weight_kg:", chartevents_pivot.shape)
chartevents_pivot.head()


✅ Chartevents pivot with unified daily_weight_kg: (1628948, 7)


label,subject_id,stay_id,charttime,cvp,end_tidal_co2,height_cm,daily_weight_kg
0,10000032,39553978,2180-07-23 12:36:00,NaN,NaN,152.0,39.40000
1,10000032,39553978,2180-07-23 14:22:00,NaN,NaN,NaN,39.32712
2,10000032,39553978,2180-07-23 14:44:00,NaN,NaN,NaN,39.32712
3,10000690,37081114,2150-11-02 18:03:00,NaN,NaN,NaN,55.30000
4,10000690,37081114,2150-11-02 20:12:00,NaN,NaN,NaN,55.15776


In [13]:
# Merge chartevents pivot into vitals
vitals_merged = vitals_merged.merge(
    chartevents_pivot,
    on=["subject_id", "stay_id", "charttime"],
    how="outer"
)

print("After adding chartevents:", vitals_merged.shape)

After adding chartevents: (13986217, 24)


### Step 6.3 Add OMR data

In [14]:
# Make sure charttime in vitals_merged is datetime
vitals_merged["charttime"] = pd.to_datetime(vitals_merged["charttime"], errors="coerce")


In [15]:
# Convert OMR chartdate → charttime with 00:00:00
omr_bp_small = omr_bp[["subject_id", "chartdate", "sbp_cuff", "dbp_cuff", "map_cuff"]].copy()
omr_bp_small["charttime"] = pd.to_datetime(omr_bp_small["chartdate"]).dt.floor("D")
omr_bp_small = omr_bp_small.drop(columns=["chartdate"])

omr_weight_small = omr_weight[["subject_id", "chartdate", "daily_weight_kg"]].copy()
omr_weight_small["charttime"] = pd.to_datetime(omr_weight_small["chartdate"]).dt.floor("D")
omr_weight_small = omr_weight_small.drop(columns=["chartdate"])

omr_height_small = omr_height[["subject_id", "chartdate", "height_cm"]].copy()
omr_height_small["charttime"] = pd.to_datetime(omr_height_small["chartdate"]).dt.floor("D")
omr_height_small = omr_height_small.drop(columns=["chartdate"])

# Merge step by step, then use combine_first to fill
vitals_merged = vitals_merged.merge(
    omr_bp_small,
    on=["subject_id", "charttime"],
    how="left",
    suffixes=('', '_omr')
)
for col in ["sbp_cuff", "dbp_cuff", "map_cuff"]:
    if f"{col}_omr" in vitals_merged.columns:
        vitals_merged[col] = vitals_merged[col].combine_first(vitals_merged[f"{col}_omr"])
        vitals_merged = vitals_merged.drop(columns=[f"{col}_omr"])

vitals_merged = vitals_merged.merge(
    omr_weight_small,
    on=["subject_id", "charttime"],
    how="left",
    suffixes=('', '_omr')
)
if "daily_weight_kg_omr" in vitals_merged.columns:
    vitals_merged["daily_weight_kg"] = vitals_merged["daily_weight_kg"].combine_first(
        vitals_merged["daily_weight_kg_omr"]
    )
    vitals_merged = vitals_merged.drop(columns=["daily_weight_kg_omr"])

vitals_merged = vitals_merged.merge(
    omr_height_small,
    on=["subject_id", "charttime"],
    how="left",
    suffixes=('', '_omr')
)
if "height_cm_omr" in vitals_merged.columns:
    vitals_merged["height_cm"] = vitals_merged["height_cm"].combine_first(
        vitals_merged["height_cm_omr"]
    )
    vitals_merged = vitals_merged.drop(columns=["height_cm_omr"])

print("After adding OMR:", vitals_merged.shape)


After adding OMR: (13986466, 24)


### Step 6.4 Add hadm_id (csn)

In [16]:
# Add hadm_id (hospital admission ID) from icustays
vitals_merged = vitals_merged.merge(
    icustays[["stay_id", "hadm_id"]],
    on="stay_id",
    how="left"
)

# Rename to pipeline-required names
vitals_merged = vitals_merged.rename(columns={
    "subject_id": "pat_id",
    "hadm_id": "csn",
    "charttime": "recorded_time"
})

print("✅ Final merge completed:", vitals_merged.shape)
vitals_merged.head()

✅ Final merge completed: (13986466, 25)


,pat_id,stay_id,recorded_time,pulse,unassisted_resp_rate,spo2,temperature,temproute,sbp_line,dbp_line,...,o2_delivery_device_1,o2_delivery_device_2,o2_delivery_device_3,o2_delivery_device_4,o2_flow_rate,cvp,end_tidal_co2,height_cm,daily_weight_kg,csn
0,10000032,39553978,2180-07-23 12:36:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,152.0,39.4,29079034
1,10000032,39553978,2180-07-23 14:00:00,NaN,NaN,NaN,37.06,Oral,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29079034
2,10000032,39553978,2180-07-23 14:11:00,NaN,NaN,NaN,NaN,NaN,84.0,48.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29079034
3,10000032,39553978,2180-07-23 14:12:00,91.0,24.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29079034
4,10000032,39553978,2180-07-23 14:13:00,NaN,NaN,98.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29079034


## Step 7. Save flat file

In [18]:
# Drop stay_id since it's not needed in the final output
if "stay_id" in vitals_merged.columns:
    vitals_merged = vitals_merged.drop(columns=["stay_id"])

# Define final desired column order
desired_columns = [
    "pat_id", "csn", "recorded_time",
    "temperature", "temproute",
    "daily_weight_kg", "height_cm",
    "sbp_line", "dbp_line", "map_line",
    "sbp_cuff", "dbp_cuff", "map_cuff",
    "pulse", "unassisted_resp_rate", "spo2", "glucose",
    "o2_delivery_device_1", "o2_delivery_device_2", "o2_delivery_device_3", "o2_delivery_device_4",
    "cvp", "end_tidal_co2",
    "o2_flow_rate"
]

# Keep only the columns that exist (in case some are missing for this cohort)
desired_columns = [col for col in desired_columns if col in vitals_merged.columns]

# Reorder dataframe
vitals_final = vitals_merged[desired_columns]

# Save to CSV
out_path = os.path.join(output_dir, "VITALS.csv")
vitals_final.to_csv(out_path, index=False)

print(f"✅ VITALS.csv saved to {out_path} with shape {vitals_final.shape}")
print("Final columns:", vitals_final.columns.tolist())
print(vitals_final.head())

✅ VITALS.csv saved to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/VITALS.csv with shape (13986466, 24)
Final columns: ['pat_id', 'csn', 'recorded_time', 'temperature', 'temproute', 'daily_weight_kg', 'height_cm', 'sbp_line', 'dbp_line', 'map_line', 'sbp_cuff', 'dbp_cuff', 'map_cuff', 'pulse', 'unassisted_resp_rate', 'spo2', 'glucose', 'o2_delivery_device_1', 'o2_delivery_device_2', 'o2_delivery_device_3', 'o2_delivery_device_4', 'cvp', 'end_tidal_co2', 'o2_flow_rate']
     pat_id       csn       recorded_time  temperature temproute  \
0  10000032  29079034 2180-07-23 12:36:00          NaN       NaN   
1  10000032  29079034 2180-07-23 14:00:00        37.06      Oral   
2  10000032  29079034 2180-07-23 14:11:00          NaN       NaN   
3  10000032  29079034 2180-07-23 14:12:00          NaN       NaN   
4  10000032  29079034 2180-07-23 14:13:00          NaN       NaN   

   daily_weight_kg  height_cm  sbp_line  dbp_line  map_line  ...  \
0             39.4      152.0  